# 06 - Async/Await & asyncio

Part of the Python, DSA & Git chapter. Threading, the previous notebook, gives concurrency for I/O-bound work by running many OS threads, each with real memory and scheduling overhead. asyncio gives the same I/O concurrency using a single thread and a single event loop, which is why it is the standard choice for serving many simultaneous requests, such as an ML model API handling thousands of concurrent inference requests, without paying for thousands of OS threads.

Covers: coroutines and async/await, the event loop, asyncio.sleep vs time.sleep, running things concurrently with gather and create_task, why asyncio outscales threading for high-concurrency I/O, async generators and async context managers, and a realistic call-many-APIs-concurrently example. Practice exercises at the end.

A notebook-specific note: this notebook runs async code using top-level await directly in cells, which Jupyter supports natively because the kernel itself already runs on an event loop. In a standalone .py script there is no event loop running yet, so the standard entry point there is asyncio.run(main()) instead -- both patterns are shown below, in Part F.

# Part A - Coroutines and the Event Loop

## async def and await: what a coroutine actually is

async def defines a coroutine function. Calling it does not run the body -- exactly like calling a generator function does not run its body either -- it returns a coroutine OBJECT that has to actually be driven to run, either with await, asyncio.run(), or by scheduling it as a task. await suspends the current coroutine until the awaited thing completes, handing control back to the event loop so it can run other work in the meantime.

In [1]:
import asyncio
import time

async def say_hello(name, delay):
    await asyncio.sleep(delay)
    return f"hello, {name}"

coro = say_hello("ada", 0.1)
print(type(coro))              # <class 'coroutine'> -- nothing has run yet

result = await coro             # NOW it actually runs -- top-level await works in Jupyter
print(result)

<class 'coroutine'>
hello, ada


## The classic gotcha: forgetting await

Calling a coroutine function without await, or without scheduling it another way, creates the coroutine object and does absolutely nothing else -- no error, just a silent no-op, which is a genuinely common and confusing bug. Python does warn about this once the unused coroutine object is garbage collected.

In [2]:
import warnings
import gc

async def do_work():
    await asyncio.sleep(0.1)
    print("work actually happened")

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    do_work()                    # MISSING await -- creates a coroutine object and discards it
    gc.collect()                  # force collection so the warning fires predictably, for this demo
    print("nothing printed above from do_work -- it never actually ran")
    for w in caught:
        print("Python warned:", w.category.__name__, "--", w.message)

nothing printed above from do_work -- it never actually ran
Python warned: RuntimeWarning -- coroutine 'do_work' was never awaited


## The event loop: single-threaded cooperative multitasking

asyncio runs everything on ONE thread, using an event loop that tracks which coroutines are ready to run and which are waiting on something, such as a sleep timer or a network response. When a coroutine hits an await on something not yet ready, it suspends, and the event loop switches to whichever OTHER coroutine is ready to make progress. This is cooperative multitasking: a coroutine has to voluntarily yield control at an await point, unlike OS threads, which the operating system can preempt at any point.

# Part B - Running Coroutines Concurrently

## asyncio.sleep vs time.sleep: the distinction that makes async work at all

time.sleep blocks the entire thread -- nothing else can run, including other coroutines, because the single event-loop thread is frozen. asyncio.sleep suspends only the current coroutine and hands control back to the event loop, which is free to run other coroutines during that wait. Using time.sleep inside async code is a common, serious mistake: it silently defeats the entire point of asyncio.

In [3]:
async def blocking_sleep_demo():
    async def bad_task(name):
        time.sleep(0.2)          # WRONG inside async code -- blocks the whole event loop
        return name

    start = time.perf_counter()
    results = await asyncio.gather(bad_task("A"), bad_task("B"), bad_task("C"))
    elapsed = time.perf_counter() - start
    print(f"time.sleep version: {elapsed:.3f}s for 3 tasks  <- ~0.6s, NOT concurrent, blocked the loop")
    return results

async def nonblocking_sleep_demo():
    async def good_task(name):
        await asyncio.sleep(0.2)  # correct -- yields control back to the event loop
        return name

    start = time.perf_counter()
    results = await asyncio.gather(good_task("A"), good_task("B"), good_task("C"))
    elapsed = time.perf_counter() - start
    print(f"asyncio.sleep version: {elapsed:.3f}s for 3 tasks  <- ~0.2s, genuinely concurrent")
    return results

await blocking_sleep_demo()
await nonblocking_sleep_demo()

time.sleep version: 0.601s for 3 tasks  <- ~0.6s, NOT concurrent, blocked the loop


asyncio.sleep version: 0.201s for 3 tasks  <- ~0.2s, genuinely concurrent


['A', 'B', 'C']

## Sequential await vs asyncio.gather

Awaiting coroutines one after another, even with proper asyncio.sleep, still runs them sequentially -- await blocks the CURRENT coroutine until that one specific thing finishes before moving to the next line. asyncio.gather schedules several coroutines to run concurrently and waits for all of them together, which is where the actual speedup comes from.

In [4]:
async def fake_api_call(request_id, delay=0.2):
    await asyncio.sleep(delay)
    return f"response for request {request_id}"

start = time.perf_counter()
results_sequential = []
for i in range(5):
    results_sequential.append(await fake_api_call(i))   # one at a time
print(f"sequential await: {time.perf_counter() - start:.3f}s for 5 calls")

start = time.perf_counter()
results_concurrent = await asyncio.gather(*(fake_api_call(i) for i in range(5)))
print(f"asyncio.gather:    {time.perf_counter() - start:.3f}s for 5 calls  <- roughly 5x faster")
print(results_concurrent)

sequential await: 1.002s for 5 calls


asyncio.gather:    0.201s for 5 calls  <- roughly 5x faster
['response for request 0', 'response for request 1', 'response for request 2', 'response for request 3', 'response for request 4']


## asyncio.create_task: starting work in the background

gather is convenient when every coroutine is already at hand, but create_task schedules a coroutine to start running immediately in the background, returning a Task object that can be awaited LATER -- useful for starting several independent pieces of work and doing something else in the meantime before collecting results.

In [5]:
async def create_task_demo():
    task_a = asyncio.create_task(fake_api_call("A", 0.2))   # starts running NOW, in the background
    task_b = asyncio.create_task(fake_api_call("B", 0.2))

    print("both tasks scheduled, doing other work while they run...")
    await asyncio.sleep(0.05)
    print("still waiting on the tasks...")

    result_a = await task_a       # if it already finished, this returns immediately
    result_b = await task_b
    return result_a, result_b

start = time.perf_counter()
print(await create_task_demo())
print(f"total: {time.perf_counter() - start:.3f}s  <- close to 0.2s, not 0.25s or more")

both tasks scheduled, doing other work while they run...
still waiting on the tasks...


('response for request A', 'response for request B')
total: 0.201s  <- close to 0.2s, not 0.25s or more


# Part C - Why asyncio Outscales Threading for High-Concurrency I/O

## The scaling argument

Each OS thread costs real resources: a default reserved stack of several megabytes on most platforms, plus real kernel scheduling overhead, so thousands of threads means gigabytes of memory spent on stacks alone, and a scheduler working harder to context-switch between all of them. An asyncio Task is a lightweight Python object, not an OS thread -- creating thousands of them costs a small, bounded amount of memory, and the single event loop switches between them far more cheaply than the OS switches between thousands of real threads. This is exactly why high-concurrency servers, such as an API serving many simultaneous inference requests, are typically built on asyncio rather than a thread pool sized in the thousands.

In [6]:
import threading

def thread_target(delay):
    time.sleep(delay)

N = 300

start = time.perf_counter()
results = await asyncio.gather(*(fake_api_call(i, delay=0.1) for i in range(N)))
asyncio_time = time.perf_counter() - start
print("asyncio,", N, "concurrent tasks:  ", round(asyncio_time, 3), "s")

start = time.perf_counter()
threads = [threading.Thread(target=thread_target, args=(0.1,)) for _ in range(N)]
for t in threads:
    t.start()
for t in threads:
    t.join()
threading_time = time.perf_counter() - start
print("threading,", N, "real OS threads:", round(threading_time, 3), "s")

print()
print("asyncio is", round(threading_time / asyncio_time, 2), "x faster at this modest N;")
print("the gap widens further at higher N, where per-thread memory and OS scheduling overhead compound.")

asyncio, 300 concurrent tasks:   0.107 s


threading, 300 real OS threads: 0.166 s

asyncio is 1.55 x faster at this modest N;
the gap widens further at higher N, where per-thread memory and OS scheduling overhead compound.


# Part D - Async Generators and Async Context Managers

## Async generators: async def with yield

An async generator combines ideas from the last two notebooks: it yields values lazily like a regular generator, but each step can also await something, such as the next chunk of a streaming API response. Consume it with async for instead of a plain for loop.

In [7]:
async def fetch_pages(n_pages):
    for page in range(1, n_pages + 1):
        await asyncio.sleep(0.05)      # simulate an awaited network call per page
        yield f"page {page} content"

async def consume_pages():
    async for page in fetch_pages(3):   # async for -- required for an async generator
        print(page)

await consume_pages()

page 1 content
page 2 content
page 3 content


## Async context managers: async with, __aenter__, __aexit__

The async equivalent of the with statement from the previous notebook, used for resources whose setup or teardown itself needs to await something, such as opening a pooled database connection or a network session. contextlib.asynccontextmanager mirrors the sync @contextmanager decorator from before.

In [8]:
from contextlib import asynccontextmanager

@asynccontextmanager
async def fake_connection_pool(name):
    print(f"acquiring connection: {name}")
    await asyncio.sleep(0.05)            # simulate an awaited setup step
    try:
        yield f"connection[{name}]"
    finally:
        print(f"releasing connection: {name}")
        await asyncio.sleep(0.02)         # simulate an awaited teardown step

async def use_connection():
    async with fake_connection_pool("db1") as conn:
        print("using", conn)

await use_connection()

acquiring connection: db1


using connection[db1]
releasing connection: db1


# Part E - Realistic Example: Calling Many APIs Concurrently, With a Concurrency Limit

## Why an unlimited gather is dangerous in real systems

Firing off asyncio.gather across, say, ten thousand requests at once will genuinely try to open ten thousand concurrent connections immediately, which can overwhelm the target server, hit rate limits, or exhaust local file descriptors. asyncio.Semaphore caps how many coroutines can be actively running a given section at once, exactly like threading.Semaphore in the previous notebook -- a standard, necessary pattern for any real call-many-APIs pipeline.

In [9]:
async def call_api_limited(request_id, semaphore, delay=0.1):
    async with semaphore:                 # blocks here if already at the concurrency limit
        await asyncio.sleep(delay)
        return f"result {request_id}"

async def call_many_apis(n_requests, max_concurrent):
    semaphore = asyncio.Semaphore(max_concurrent)
    tasks = [call_api_limited(i, semaphore) for i in range(n_requests)]
    return await asyncio.gather(*tasks)

start = time.perf_counter()
results = await call_many_apis(n_requests=50, max_concurrent=10)
elapsed = time.perf_counter() - start
print(f"50 requests, max 10 concurrent: {elapsed:.3f}s")
print(f"expected roughly: {50 / 10 * 0.1:.2f}s  (50 requests / 10 at a time * 0.1s each)")
print(len(results), "results collected")

50 requests, max 10 concurrent: 0.502s
expected roughly: 0.50s  (50 requests / 10 at a time * 0.1s each)
50 results collected


## Handling failures inside gather

By default, if any coroutine passed to gather raises, gather immediately raises that exception and cancels the rest, which is often not what is wanted when calling many independent APIs, since one failure should not necessarily discard forty-nine successful results. return_exceptions=True instead collects exceptions as regular values in the results list, so the caller can inspect and handle them individually.

In [10]:
async def flaky_api_call(request_id):
    await asyncio.sleep(0.05)
    if request_id % 7 == 0:
        raise ConnectionError(f"request {request_id} failed")
    return f"result {request_id}"

results = await asyncio.gather(
    *(flaky_api_call(i) for i in range(10)),
    return_exceptions=True,
)
for r in results:
    if isinstance(r, Exception):
        print("failed:", r)
    else:
        print("ok:", r)

failed: request 0 failed
ok: result 1
ok: result 2
ok: result 3
ok: result 4
ok: result 5
ok: result 6
failed: request 7 failed
ok: result 8
ok: result 9


# Part F - The Standalone-Script Entry Point: asyncio.run(main())

Everything above used top-level await, which only works because Jupyter already has an event loop running. In a normal .py script there is no event loop yet, so the standard pattern is to put all the async logic inside one coroutine, commonly named main, and start the ENTIRE program with asyncio.run(main()) exactly once, at the very top level of the script. This exact cell is not executed here, since calling asyncio.run() inside a notebook that already has a running loop raises a RuntimeError -- this is precisely why the notebook uses top-level await instead, throughout.

```python
import asyncio

async def fetch_all(urls):
    async def fetch_one(url):
        await asyncio.sleep(0.1)     # a real version would use aiohttp or httpx here
        return f"fetched {url}"
    return await asyncio.gather(*(fetch_one(u) for u in urls))

async def main():
    results = await fetch_all(["url1", "url2", "url3"])
    print(results)

if __name__ == "__main__":
    asyncio.run(main())    # the ONE place asyncio.run is called, at the very top level
```

## Practice exercises

Implement each TODO, then run the check cell. Use top-level await when running your solutions, consistent with the rest of this notebook.

In [11]:
async def fetch_all_concurrent(items, delay=0.05):
    # "Fetch" each item concurrently (await asyncio.sleep(delay) per item) and return a list
    # of "fetched {item}" strings in the SAME order as `items`.
    # TODO: implement using asyncio.gather
    raise NotImplementedError


async def run_with_limit(coro_factories, max_concurrent):
    # coro_factories is a list of zero-arg functions, each of which returns a NEW coroutine
    # when called. Run all of them concurrently but respect max_concurrent using a Semaphore.
    # Return the list of results in the SAME order as coro_factories.
    # TODO: implement using asyncio.Semaphore + asyncio.gather
    raise NotImplementedError

In [12]:
def _check(label, ok, detail=""):
    print("  [" + ("PASS" if ok else "FAIL") + "]", label, detail)

try:
    result = await fetch_all_concurrent(["a", "b", "c"], delay=0.02)
    _check("fetch_all_concurrent preserves order", result == ["fetched a", "fetched b", "fetched c"], result)
except NotImplementedError:
    print("  [SKIP] fetch_all_concurrent -- not implemented yet")
except Exception as e:
    print("  [ERROR] fetch_all_concurrent --", e)

try:
    async def make_task(i):
        await asyncio.sleep(0.01)
        return i * i
    factories = [(lambda i=i: make_task(i)) for i in range(5)]
    result = await run_with_limit(factories, max_concurrent=2)
    _check("run_with_limit preserves order and respects the limit", result == [0, 1, 4, 9, 16], result)
except NotImplementedError:
    print("  [SKIP] run_with_limit -- not implemented yet")
except Exception as e:
    print("  [ERROR] run_with_limit --", e)

  [SKIP] fetch_all_concurrent -- not implemented yet
  [SKIP] run_with_limit -- not implemented yet


## Self-check before moving on

- [ ] I can explain what a coroutine is and why calling an async function does not run it
- [ ] I can explain the missing-await bug and why Python only warns rather than errors
- [ ] I can explain why time.sleep inside async code is a serious bug, and what asyncio.sleep does differently
- [ ] I can use asyncio.gather to run several coroutines concurrently, and explain how that differs from sequential await
- [ ] I can explain, with a number attached, why asyncio scales to more concurrent I/O tasks than one-thread-per-task
- [ ] I can write an async generator using async def and yield, and consume it with async for
- [ ] I can write an async context manager with @asynccontextmanager
- [ ] I know to use asyncio.Semaphore to cap concurrency in a real call-many-APIs pipeline
- [ ] I know return_exceptions=True exists for handling partial failures in gather
- [ ] I know the standalone-script entry point is asyncio.run(main()), called once, not the top-level await used in this notebook

This closes out the core Concurrency block: threading and asyncio together. Next up in the Python subchapter: `07-apis-requests-fastapi-for-ml.ipynb`, building on both of these to actually serve an ML model behind an endpoint.